# Experiment 2-S — Does the Stopping Rule Actually Work? (NBA domain)

**Claims under test: C1, C2** (webaggr.pdf §3.2–3.3, Theorem 1, Assumption 1(a)).
Contract: `experiments/exp2s/PREREGISTRATION_EXP2S.md` — read it first; nothing
below may change after the first live run without a dated addendum.

Design (mirrors the finance Exp 2 full-cohort notebook):
- **LIVE**: one run per cohort player at ε = 0.05, δ_M = 0.05, $5 search budget,
  behind a `RUN_LIVE` spend gate with an idempotent per-player runs index
  (a finished player is never re-rolled).
- **OFFLINE**: arms ε ∈ {0.30, 0.20, 0.10} replayed from the recorded ledgers —
  valid because ε is read only in the stop test.
- **Self-tests S1/S2** must pass before any table or plot is produced.
- **Reporting split** (pre-registered): statistical-path vs checksum-path
  certifications reported separately; the C1/C2 headline uses statistical only.

> ⚠️ **Before the live cell**: reconcile `webaggs/frontier.py::psi` with the
> production psi in the finance repo. Under the impl guide's formula as written,
> ψ ≈ Y_CAP(1+β)·√(2·max_steps·ln(1/(δ_M·w_g)))/n, which dwarfs ε at NBA career
> sizes (n ≤ ~25) and makes the statistical certificate unreachable by
> arithmetic — bad outcome #3 by construction, not by data.

In [ ]:
# Cell 1 — setup. os.chdir to the repo root BEFORE any webaggs import:
# the CWD-relative trap crashed finance runs mid-spend (methodology notes).
import os, sys, json, csv
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "webaggs").exists():          # notebook may start in notebooks/
    if REPO == REPO.parent:
        raise RuntimeError("Could not locate the repo root (webaggs/ not found)")
    REPO = REPO.parent
os.chdir(REPO); sys.path.insert(0, str(REPO))

from webaggs import config
from webaggs.replay import recall_at_step, recall_trajectory, replay_stop
from webaggs.storage import Ledger
from webaggs.truth import grade, load_truth

# --- experiment constants (pre-registered; do not edit after the first run) ---
COHORT     = os.environ.get("WEBAGG_COHORT", "nba_v1")  # env override for pilots
LIVE_EPS   = 0.05
LIVE_DELTA = 0.05
EPS_ARMS   = (0.30, 0.20, 0.10, 0.05)           # 0.05 = the live arm
BUDGET_USD = 5.0

EXP_DIR  = config.DATA_DIR / "experiments" / "exp2s"
RUNS_DIR = EXP_DIR / "runs"
RES_DIR  = EXP_DIR / "results"; RES_DIR.mkdir(parents=True, exist_ok=True)
IDX_PATH = EXP_DIR / "runs_index.json"

def runs_index():
    return json.loads(IDX_PATH.read_text(encoding="utf-8")) if IDX_PATH.exists() else {}

def save_index(idx):
    EXP_DIR.mkdir(parents=True, exist_ok=True)
    IDX_PATH.write_text(json.dumps(idx, indent=2), encoding="utf-8")

def write_csv(path, rows):
    if not rows:                                 # loud, never a crash on empties
        print(f"[warn] no rows for {path} — wrote empty file"); Path(path).write_text(""); return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

def run_id_of(player):                           # shared with scripts/run_exp2s.py
    return f"exp2s_{COHORT}_{player.replace(' ', '_')}"

print(f"repo: {REPO}")
print(f"live arm: eps={LIVE_EPS}, delta_M={LIVE_DELTA}; replay arms: {EPS_ARMS}")

In [ ]:
# Cell 2 — preflight. Loud refusals, no silent degradation:
# * truth table must exist AND match its freeze manifest (sha256);
# * the conformal gate refuses cold starts (no calibration file -> raise).
truth = load_truth(config.DATA_DIR, COHORT)      # verifies the freeze hash
print(f"cohort '{COHORT}': {len(truth)} players, "
      f"{sum(len(s) for s in truth.values())} truth (player, season) records")

from webaggs.conformal import ConformalGate
gate = ConformalGate.load()                      # raises if unfitted
print(f"conformal gate fitted: threshold={gate.threshold:.3f} on n_cal={gate.n_cal}")

idx = runs_index()
done = sorted(p for p, m in idx.items() if m.get("done"))
print(f"finished runs: {len(done)}/{len(truth)}")

## Live phase

One run per player at the pre-registered live arm. Gated: refuses unless
`RUN_LIVE=1` **and** both API keys are set, and the pytest suite is the gate
before any spend (run it in a terminal first). Idempotent: finished players are
skipped — never re-rolled after seeing results.

In [ ]:
# Cell 3 — LIVE (spend). Everything real: live search, live fetches with
# the oracle denied inside fetch_url, real LLM extraction. No fakes in any
# experiment path.
if not config.live_enabled():
    print("RUN_LIVE != 1 — live phase skipped. Set RUN_LIVE=1 after pytest is green.")
else:
    for env in (config.OPENAI_API_KEY_ENV, config.SERPER_API_KEY_ENV):
        assert os.environ.get(env), f"{env} not set — refusing to run."
    from webaggs.pipeline import end_to_end
    from webaggs.search import SerperBackend
    from webaggs.storage import fresh_ledger
    from webaggs.truth import load_display_names
    display = load_display_names(config.DATA_DIR, COHORT)   # search with real casing
    search = SerperBackend()
    idx = runs_index()
    for player in sorted(truth):
        if idx.get(player, {}).get("done"):
            print(f"[skip] {player} already run"); continue
        rid = run_id_of(player)
        # a crashed (unfinished) ledger is moved aside — never append a second
        # trajectory under one run_id (it would corrupt every offline replay)
        ledger = fresh_ledger(RUNS_DIR, rid)
        print(f"[live] {player} at eps={LIVE_EPS} ...")
        ans = end_to_end(display.get(player, player), rid, ledger, search=search, gate=gate,
                         eps_g=LIVE_EPS, delta_m=LIVE_DELTA, budget_usd=BUDGET_USD)
        ledger.close()
        idx[player] = {"done": True, "run_id": rid, "stop_reason": ans.stop_reason,
                       "certificate": ans.certificate, "total": ans.total,
                       "n_records": ans.n_records}
        save_index(idx)                           # resume point after every player
        print(f"    -> {ans.stop_reason} / {ans.certificate}: "
              f"{ans.n_records} records, total {ans.total}")
    done = sorted(p for p, m in idx.items() if m.get("done"))
    print(f"live phase: {len(done)}/{len(truth)} players finished")

## Self-tests (must pass before any result below is read)

- **S1** — replay at the live ε reproduces the live certified stop step for
  every run that stopped `certified`.
- **S2** — replayed stop steps are monotone non-increasing as ε loosens
  (and a stop at a stricter ε implies a stop at every looser ε).

A failure here aborts the notebook: the ledgers do not reproduce the live
decision and nothing downstream may be reported until it is diagnosed.

In [ ]:
# Cell 4 — load ledgers once, then S1/S2 as hard asserts.
idx = runs_index()
runs = {}                                        # player -> (ms, cov, live_stop)
for player, meta in sorted(idx.items()):
    if not meta.get("done"):
        continue
    led = Ledger(RUNS_DIR / f"{meta['run_id']}.sqlite")
    runs[player] = (led.measurements(meta["run_id"], metric="U_hat"),
                    led.coverage(meta["run_id"]),
                    led.stop(meta["run_id"]))
    led.close()
assert runs, "No finished runs — execute the live cell first."

for player, (ms, cov, live) in runs.items():
    if live and live[1] == "certified":          # S1
        rr = replay_stop(ms, LIVE_EPS)
        assert rr.stop_step == live[0], (
            f"S1 FAILED for {player}: live stop {live[0]}, replay {rr.stop_step}")
    prev = None
    for eps in sorted(EPS_ARMS):                 # strictest first — S2
        rr = replay_stop(ms, eps)
        if prev is not None:
            assert rr.stop_step is not None and rr.stop_step <= prev, (
                f"S2 FAILED for {player} at eps={eps}")
        prev = rr.stop_step if rr.stop_step is not None else prev
print(f"Self-tests S1/S2 passed over {len(runs)} runs.")

## Replay arms → stop table

One row per (player, ε): stop step, stop kind (**statistical / checksum /
censored**), spend and realized record recall at the stop, and whether the
promise `recall ≥ 1 − ε` was violated. Checksum stops are flagged and excluded
from the statistical headline (pre-registered split).

In [ ]:
# Cell 5 — replay every arm from the frozen ledgers (no fetches, no spend).
stops_rows = []
for player, (ms, cov, live) in runs.items():
    truth_keys = {f"{player}|{s}" for s in truth[player]}
    for eps in EPS_ARMS:
        rr = replay_stop(ms, eps)
        if rr.stop_step is not None:
            row = next(r for r in ms if r[0] == rr.stop_step)
            kind = "checksum" if str(row[4].get("certified") or "") else "statistical"
            spend = row[4].get("spent", 0.0)
        else:
            kind, spend = "censored", (ms[-1][4].get("spent", 0.0) if ms else 0.0)
        rec = recall_at_step(cov, truth_keys, rr.stop_step)
        stops_rows.append({"player": player, "eps": eps,
                           "stop_step": rr.stop_step if rr.stop_step is not None else "",
                           "stop_kind": kind, "censored": int(rr.censored),
                           "spend_at_stop": spend,
                           "recall_at_stop": round(rec, 4),
                           "promised": 1 - eps,
                           "violated": int(rec < 1 - eps)})
write_csv(RES_DIR / "stops.csv", stops_rows)

hdr = ("player", "eps", "stop_step", "stop_kind", "recall_at_stop", "violated")
print(" | ".join(f"{h:>16}" for h in hdr))
for r in stops_rows:
    print(" | ".join(f"{str(r[h]):>16}" for h in hdr))

## Plot 1 — C1 calibration (the plot the whole stopping story stands on)

Claimed unseen mass (Û + ψ, capped at 1) vs realized unseen mass
(1 − recall) at every logged step. **Any point above the diagonal falsifies
Assumption 1(a).** Raw Û is shown faint: the finance pilot saw raw Û drift
optimistic while Û + ψ held — the same decomposition is worth seeing here.

In [ ]:
# Cell 6 — C1 calibration points + plot.
import matplotlib.pyplot as plt

cal_rows = []
for player, (ms, cov, _live) in runs.items():
    truth_keys = {f"{player}|{s}" for s in truth[player]}
    traj = dict(recall_trajectory(cov, truth_keys))
    running = 0.0
    for step, _g, _m, u, extra in ms:
        running = traj.get(step, running)
        cal_rows.append({"player": player, "step": step,
                         "claimed_unseen": round(min(u + extra["psi"], 1.0), 4),
                         "raw_U_hat": round(min(u, 1.0), 4),
                         "realized_unseen": round(1 - running, 4)})
write_csv(RES_DIR / "calibration.csv", cal_rows)

above = [r for r in cal_rows if r["claimed_unseen"] < r["realized_unseen"]]
x  = [r["realized_unseen"] for r in cal_rows]
y  = [r["claimed_unseen"] for r in cal_rows]
yr = [r["raw_U_hat"] for r in cal_rows]
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="diagonal (claimed = realized)")
ax.scatter(x, yr, s=12, alpha=0.25, label="raw $\\hat{U}$")
ax.scatter(x, y,  s=14, alpha=0.6,  label="claimed $\\hat{U}+\\psi$ (cap 1)")
ax.set_xlabel("realized unseen mass (1 − recall)")
ax.set_ylabel("claimed unseen mass")
ax.set_title(f"C1 calibration — {len(above)}/{len(cal_rows)} points above the diagonal")
ax.legend(); ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.show()
print(f"C1: {len(above)}/{len(cal_rows)} certificate points above the diagonal"
      " — any such point falsifies Assumption 1(a).")

## Plot 2 — C2 coverage (realized recall at the stop vs the promise)

Per arm: realized recall at the replayed stop vs the promised 1 − ε.
**Headline counts use the statistical path only**; checksum and censored stops
are shown but set aside (the exp3 p3 lesson: a checksum can certify with most
of the group undiscovered, bypassing the statistical test entirely).

In [ ]:
# Cell 7 — C2 coverage plot + violation table (statistical path only).
fig, ax = plt.subplots(figsize=(7, 5))
markers = {"statistical": "o", "checksum": "s", "censored": "x"}
for kind, mk in markers.items():
    pts = [r for r in stops_rows if r["stop_kind"] == kind]
    if pts:
        ax.scatter([r["promised"] for r in pts],
                   [r["recall_at_stop"] for r in pts],
                   marker=mk, s=40, alpha=0.7, label=kind)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("promised recall (1 − ε)"); ax.set_ylabel("realized recall at stop")
ax.set_title("C2 — recall at stop vs promise (points below the line violate)")
ax.legend(); ax.set_xlim(0.6, 1.01); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

print(f"{'eps':>6} | {'stat stops':>10} | {'violations':>10} | "
      f"{'checksum':>8} | {'censored':>8}")
for eps in EPS_ARMS:
    arm = [r for r in stops_rows if r["eps"] == eps]
    stat = [r for r in arm if r["stop_kind"] == "statistical"]
    vio  = sum(r["violated"] for r in stat)
    print(f"{eps:>6} | {len(stat):>10} | {vio:>10} | "
          f"{sum(r['stop_kind']=='checksum' for r in arm):>8} | "
          f"{sum(r['stop_kind']=='censored' for r in arm):>8}")
stat_live = [r for r in stops_rows
             if r["eps"] == LIVE_EPS and r["stop_kind"] == "statistical"]
print(f"\nHEADLINE (statistical path, eps={LIVE_EPS}): "
      f"{len(stat_live)} certified stops, "
      f"{sum(r['violated'] for r in stat_live)} violations "
      f"(delta_M budget = {LIVE_DELTA}).")

## Shortfall decomposition (the Fig-5 analogue)

Per player at the live arm: truth seasons missed at the stop, split into
**lost-to-stopping-early** (discovered later in the recorded trajectory) vs
**never-discovered** (absent from the whole recording). In finance,
never-discovered dominating + the paywall audit was the access-censoring
story; here the accessible-web ceiling is ~100%, so a never-discovered pile
points at query formulation / frontier depth instead.

In [ ]:
# Cell 8 — shortfall decomposition at the live arm + stacked bars.
short_rows = []
for player, (ms, cov, _live) in runs.items():
    truth_keys = {f"{player}|{s}" for s in truth[player]}
    for eps in EPS_ARMS:
        rr = replay_stop(ms, eps)
        by_stop = {rk for s, rk in cov if rk in truth_keys
                   and (rr.stop_step is None or s <= rr.stop_step)}
        ever = {rk for _s, rk in cov if rk in truth_keys}
        short_rows.append({"player": player, "eps": eps,
                           "n_truth": len(truth_keys),
                           "lost_to_stopping": len(ever - by_stop),
                           "never_discovered": len(truth_keys - ever)})
write_csv(RES_DIR / "shortfall.csv", short_rows)

live_rows = sorted((r for r in short_rows if r["eps"] == LIVE_EPS),
                   key=lambda r: r["player"])
names = [r["player"] for r in live_rows]
lost  = [r["lost_to_stopping"] for r in live_rows]
never = [r["never_discovered"] for r in live_rows]
fig, ax = plt.subplots(figsize=(max(7, 0.4 * len(names)), 4.5))
ax.bar(names, lost, label="lost to stopping early")
ax.bar(names, never, bottom=lost, label="never discovered")
ax.set_ylabel("truth seasons missed")
ax.set_title(f"Shortfall decomposition at eps={LIVE_EPS}")
ax.legend(); plt.xticks(rotation=75, ha="right")
plt.tight_layout(); plt.show()

## Exact grading vs the frozen key

Record recall (season found — what the stopping rule certifies), exact recall
(season *and* points right — the value side), spurious count and relative
aggregation error, with the certificate path for the split.

In [ ]:
# Cell 9 — exact grading of the live runs' resolved records.
grade_rows = []
for player, meta in sorted(idx.items()):
    if not meta.get("done"):
        continue
    led = Ledger(RUNS_DIR / f"{meta['run_id']}.sqlite")
    asserted = {season: value for _rk, season, value, _b, _d
                in led.resolved(meta["run_id"])}
    led.close()
    g = grade(player, truth[player], asserted)
    grade_rows.append({"player": player, "certificate": meta["certificate"],
                       "stop_reason": meta["stop_reason"],
                       "n_truth": g.n_truth, "n_asserted": g.n_asserted,
                       "record_recall": round(g.record_recall, 4),
                       "exact_recall": round(g.exact_recall, 4),
                       "n_spurious": g.n_spurious,
                       "true_total": g.true_total,
                       "asserted_total": g.asserted_total,
                       "rel_agg_error": round(g.rel_agg_error, 4)})
write_csv(RES_DIR / "grades.csv", grade_rows)

hdr = ("player", "certificate", "record_recall", "exact_recall",
       "n_spurious", "rel_agg_error")
print(" | ".join(f"{h:>14}" for h in hdr))
for r in grade_rows:
    print(" | ".join(f"{str(r[h]):>14}" for h in hdr))
n = len(grade_rows)
print(f"\nmeans over {n} players: record_recall "
      f"{sum(r['record_recall'] for r in grade_rows)/n:.3f}, exact_recall "
      f"{sum(r['exact_recall'] for r in grade_rows)/n:.3f}, rel_agg_error "
      f"{sum(r['rel_agg_error'] for r in grade_rows)/n:.3f}")

## Wrap-up — what goes to the supervisor

1. The C1 plot and the above-diagonal count (the direct audit of Assumption 1(a)).
2. The C2 table, statistical path only, with the checksum/censored columns beside it.
3. The shortfall decomposition — with the access ceiling removed, which failure
   mode remains?
4. The pre-registered bad-outcome checklist (prereg §9): tick what happened;
   failures are findings, not re-rolls.

All tables are also under `data/experiments/exp2s/results/` (same files the
standalone scripts write, so either entry point works).